In [ ]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
importlib.reload(kinematics)
from kinematics import azaltroll_to_theta,apply_mechanical_corrections, q_from_azaltroll, MountModelParams


In [ ]:
csv_filename = './n522_extract.csv'
d = pd.read_csv(csv_filename)
d.describe()
d['dev_p_theta1'] = ((d['s_theta1'] - d['p_theta1'] + 180) % 360 - 180)*60
d['dev_p_theta2'] = ((d['s_theta2'] - d['p_theta2'] + 180) % 360 - 180)*60
d['dev_p_theta3'] = ((d['s_theta3'] - d['p_theta3'] + 180) % 360 - 180)*60
if 'm_theta1' in d.columns:
    d['dev_m_theta1'] = ((d['s_theta1'] - d['m_theta1'] + 180) % 360 - 180)*60
    d['dev_m_theta2'] = ((d['s_theta2'] - d['m_theta2'] + 180) % 360 - 180)*60
    d['dev_m_theta3'] = ((d['s_theta3'] - d['m_theta3'] + 180) % 360 - 180)*60
d['g_az'] = np.round(d['p_az'] / 5) * 5 
d['g_alt'] = np.round(d['p_alt'] / 5) * 5
d['g_roll'] = np.round(d['p_roll'] / 5) * 5
d.columns

In [ ]:
if 'm_theta1' in d.columns:
    d[['m_az', 'm_alt', 'm_roll', 'm_theta1', 'm_theta2', 'm_theta3','dev_m_theta1','dev_m_theta2','dev_m_theta3'  ]].corr(numeric_only=True)

# Review of Collected Data and Residuals

In [ ]:
fig = go.Figure()
legend_offset = 400
d['legend_az'] = d['p_az']/360*100 + legend_offset
d['legend_alt'] = d['p_alt'] + legend_offset
d['legend_roll'] = d['p_roll'] + legend_offset
xfield='date_obs'
for yfield in ['dev_p_az','dev_p_alt','dev_p_roll', 'legend_az', 'legend_alt', 'legend_roll']:
    fig.add_trace(go.Scatter(
        x=d[xfield], y=d[yfield], 
        mode='markers+lines', name=yfield,
        marker=dict(size=6),
    ))

fig.update_layout(
    title=dict(text=f'Axis Residuals vs {xfield}', x=0.5, font=dict(size=24, family='Arial')),
    xaxis_title=f'{xfield}', yaxis_title=f'Axis Residuals (arc-min)', 
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    hovermode='x unified',
    height=800, width=1400
)

fig.show()

# Theta Space Residuals - BEFORE

In [ ]:
fig = px.scatter_matrix(d, dimensions=["dev_p_theta1", "dev_p_theta2", "dev_p_theta3", "p_theta1", "p_theta2", "p_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Theta Space Residuals - AFTER

In [ ]:
fig = px.scatter_matrix(d, dimensions=["dev_m_theta1","dev_m_theta2", "dev_m_theta3", "m_theta1", "m_theta2", "m_theta3"], color="m_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - BEFORE

In [ ]:
fig = px.scatter_matrix(d, dimensions=["dev_p_az", "dev_p_alt", "dev_p_roll", "p_theta1", "p_theta2", "p_theta3"], color="p_alt")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - AFTER

In [ ]:
fig = px.scatter_matrix(d, dimensions=["dev_m_az", "dev_m_alt", "dev_m_roll", "m_theta1", "m_theta2", "m_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

#  VII. Mechnical Correction (arcmin) by Roll and Altitude

In [ ]:
params = MountModelParams.from_config({
    "m3_tilt_alt": -2.1048,
    "m3_tilt_az": +1.5241,
    "m2_tilt_alt_amp": +67.11,
    "m2_tilt_alt_zero": 0.0,
    "m3_encoder_scale": 0.0   
})

for alt in range(70,-1,-10):
    s = f"**Altitude {alt:2.0f}°**   | "
    for roll in range(-70, +71, 10):
        q = q_from_azaltroll(180,alt,roll)
        q_adj, mag = apply_mechanical_corrections(q,params)
        mag = mag*60
        s = s + f"{mag:5.0f} | "
    print(s)


# Scrap Area


In [ ]:
fig = px.scatter(d, x="p_theta3", y="dev_p_az", color="p_theta2")
fig.show()